In [1]:
# Student Name: Mohammad Abdalrouf Alnaghmoush
# Student ID: 2230004028

# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [2]:
!pip -q install transformers sentencepiece torch



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [3]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [4]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/transformers/generation/utils.py:1375: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [6]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [7]:
# Task 1 — Model Families

tasks = {
    "Sentiment classification": "Encoder-only",
    "Machine translation": "Encoder-decoder",
    "Open-ended text generation": "Decoder-only",
    "Named Entity Recognition": "Encoder-only",
    "Text summarization": "Encoder-decoder"
}

for task, family in tasks.items():
    print(f"{task}: {family}")


Sentiment classification: Encoder-only
Machine translation: Encoder-decoder
Open-ended text generation: Decoder-only
Named Entity Recognition: Encoder-only
Text summarization: Encoder-decoder


## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [8]:
# Task 2 — Encoder-only Model

sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers."
]

results = classifier(sentences)

for sentence, result in zip(sentences, results):
    print(f"Sentence: {sentence}")
    print(f"Label: {result['label']}")
    print(f"Score: {result['score']:.4f}")
    print("-" * 50)


Sentence: The movie was excellent.
Label: POSITIVE
Score: 0.9999
--------------------------------------------------
Sentence: The lecture was difficult to understand.
Label: NEGATIVE
Score: 0.9997
--------------------------------------------------
Sentence: I enjoyed learning about transformers.
Label: POSITIVE
Score: 0.9997
--------------------------------------------------


## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [9]:
# Task 3 — Encoder–Decoder Model

# The lab does not provide a paragraph, so this short example paragraph is used.
paragraph = """
Natural language processing allows computers to understand and work with human language.
Modern NLP systems use transformer models for tasks such as translation, summarization,
sentiment analysis, and question answering. These models have greatly improved the
performance of many language-based applications.
"""

prompt = "Summarize: " + paragraph

inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

summary_ids = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Original paragraph:")
print(paragraph)

print("\nSummary:")
print(summary)


Original paragraph:

Natural language processing allows computers to understand and work with human language.
Modern NLP systems use transformer models for tasks such as translation, summarization,
sentiment analysis, and question answering. These models have greatly improved the
performance of many language-based applications.


Summary:
Learn how to use a transformer.


## Task 5 — Short Reflection

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**

A decoder-only model is suitable for a chatbot because it is designed to generate text one token at a time based on the previous context. It can use the conversation history as context and predict a relevant continuation. This makes it effective for producing natural, open-ended responses. It is also well suited for multi-turn conversation and text completion tasks.


# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [10]:
# Optional Challenge

prompt = "Natural language processing is"

for i in range(3):
    output = generator(
        prompt,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.9
    )
    print(f"Generation {i + 1}:")
    print(output[0]["generated_text"])
    print("-" * 60)


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Generation 1:
Natural language processing is the biggest industry in terms of the number of languages used or used by the government. The language processing of languages is the biggest industry in terms of the
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Generation 2:
Natural language processing is very difficult, but it always has to be done fairly. It›s difficult to translate in many languages. The most popular way to look at
------------------------------------------------------------


Generation 3:
Natural language processing is the most widely used technique for processing large numbers of text messages. This is the most commonly used form of processing text messages.





------------------------------------------------------------


### Optional Challenge Answer

The outputs can be different because `do_sample=True` makes the model sample the next token from a probability distribution instead of always choosing the single most likely token. Sampling introduces randomness, so multiple runs with the same prompt can follow different token choices. Parameters such as `temperature` also affect how varied the generated text can be.
